# Six-way factor benchmark — SemanticSCVI (Geneformer) vs SemanticSCVI (GenePT) vs LDVAE+ vs scHPF vs cNMF vs EXPIMAP

Trains the six models on `monocytes_clean.h5ad` (20 000 classical monocytes, 4 032 Ensembl-keyed genes), runs the full `SemanticBenchmark` figure + LLM-grading pipeline against `lib1_immune.gmt` and `lib2_monocyte.gmt`, and writes a single self-contained `report.html`.

Both SemanticSCVI variants share the architecture/loss family but use different gene-embedding priors:
- `semantic_geom`   — Geneformer V2 (1152-d gc104M token embeddings)
- `semantic_genept` — GenePT text-embedding-3-large (3072-d NCBI+UniProt summaries, Zenodo 10833191)

The sixth model — `expimap_k10` (scArches EXPIMAP) — uses a **pathway mask** instead of a gene-embedding prior. The mask is built from 10 hand-picked monocyte-relevant HALLMARK terms (symbol→Ensembl translated from `lib1_immune.gmt`), giving 10 latent gene programs to match `N_LATENT=10` across the other models.

All architecture / training knobs are explicit in **Cell 2 (parameters)** — separate kwargs blocks per prior, everything else shared.

In [ ]:
# ============================================================
# Parameters — edit here. All training/architecture knobs in one place.
# ============================================================
import hashlib
import json
from pathlib import Path


def _find_nb_dir() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        for cand in (
            p / "notebooks" / "benchmark_helpers.py",
            p / "benchmark_helpers.py",
            p / "scvi-tools-neural-nmf" / "notebooks" / "benchmark_helpers.py",
        ):
            if cand.exists():
                return cand.parent.resolve()
    raise FileNotFoundError(
        f"Could not locate benchmark_helpers.py from cwd={Path.cwd()}. "
        "Launch jupyter under the scvi-tools-neural-nmf tree, or set NB_DIR manually."
    )


def _semantic_cache_slug(kwargs, max_epochs, warmup_epochs, n_epochs_kl_warmup, hvg_top_n, n=10):
    """Stable 10-char hash of every param that affects the trained SemanticSCVI model.
    Change any of these and the cache_dir flips -> fresh train. Same params -> cache hit."""
    blob = json.dumps(
        {
            "kwargs": dict(sorted(kwargs.items())),
            "max_epochs": max_epochs,
            "warmup_epochs": warmup_epochs,
            "n_epochs_kl_warmup": n_epochs_kl_warmup,
            "hvg_top_n": hvg_top_n,
        },
        default=str, sort_keys=True,
    )
    return hashlib.sha1(blob.encode()).hexdigest()[:n]


NB_DIR = _find_nb_dir()
print(f"NB_DIR = {NB_DIR}")

ADATA_PATH = NB_DIR / "monocytes_clean.h5ad"
# Geneformer prior (existing).
SEMANTIC_CACHE_GENEFORMER = NB_DIR / "monocytes_clean_geneformer.pt"
# GenePT prior — raw pickle (downloaded on first call) + per-adata aligned tensor cache.
SEMANTIC_CACHE_GENEPT     = NB_DIR / "monocytes_clean_genept_3large.pt"
GENEPT_PICKLE_PATH        = NB_DIR / "GenePT_gene_protein_embedding_model_3_text.pickle"

PATHWAY_INDEX = NB_DIR / "pathway_index.pkl"
LIB1_GMT = NB_DIR / "lib1_immune.gmt"
LIB2_GMT = NB_DIR / "lib2_monocyte.gmt"
OUT_DIR = NB_DIR / "benchmark_results" / "four_way"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preprocessing ----
HVG_TOP_N = None       # None = use all genes; int = take top-N HVGs (sc.pp.highly_variable_genes)
HVG_FLAVOR = "seurat_v3"  # works on raw counts; switch to "seurat" / "cell_ranger" if you log-normalized

# ---- Cache root ----
MODEL_CACHE_DIR = NB_DIR / ".model_cache"

# ---- Shared model size ----
N_LATENT = 10

# ---- SemanticSCVI (Geneformer prior) ----
# use_batch_norm flows into BOTH encoder and decoder on the upstream LDVAE.
SEMANTIC_GENEFORMER_MAX_EPOCHS = 250
SEMANTIC_GENEFORMER_WARMUP_EPOCHS = 20
SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP = 160  # linear KL anneal 0→1 over N epochs
SEMANTIC_GENEFORMER_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=2000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- SemanticSCVI (GenePT prior) — independent knobs ----
SEMANTIC_GENEPT_MAX_EPOCHS = 250
SEMANTIC_GENEPT_WARMUP_EPOCHS = 40
SEMANTIC_GENEPT_N_EPOCHS_KL_WARMUP = 160
SEMANTIC_GENEPT_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=2000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- Param-aware cache_dirs ----
# Each SemanticSCVI variant gets its own subdir keyed on (kwargs, epochs, warmup, kl-warmup, hvg).
# Change any of those upstream -> fresh slug -> cache miss -> retrain.
# Same params on a rerun -> cache hit, no retraining.
SEMANTIC_GENEFORMER_CACHE_DIR = MODEL_CACHE_DIR / "semantic_scvi" / _semantic_cache_slug(
    SEMANTIC_GENEFORMER_KWARGS,
    SEMANTIC_GENEFORMER_MAX_EPOCHS,
    SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    HVG_TOP_N,
)
SEMANTIC_GENEPT_CACHE_DIR = MODEL_CACHE_DIR / "semantic_scvi_genept" / _semantic_cache_slug(
    SEMANTIC_GENEPT_KWARGS,
    SEMANTIC_GENEPT_MAX_EPOCHS,
    SEMANTIC_GENEPT_WARMUP_EPOCHS,
    SEMANTIC_GENEPT_N_EPOCHS_KL_WARMUP,
    HVG_TOP_N,
)
print(f"semantic_geom   cache_dir: {SEMANTIC_GENEFORMER_CACHE_DIR}")
print(f"semantic_genept cache_dir: {SEMANTIC_GENEPT_CACHE_DIR}")

# ---- Force-retrain switches (override the cache check even if params match) ----
FORCE_TRAIN_SEMANTIC_GENEFORMER = False
FORCE_TRAIN_SEMANTIC_GENEPT     = False
FORCE_TRAIN_LDVAE = False
FORCE_TRAIN_SCHPF = False
FORCE_TRAIN_NMF = False
FORCE_TRAIN_EXPIMAP = False

# ---- LDVAE+ training ----
LDVAE_MAX_EPOCHS = 250
LDVAE_KWARGS = dict(
    n_hidden=128,
    n_latent=N_LATENT,
    n_layers=1,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- scHPF / cNMF ----
N_FACTORS = N_LATENT
NMF_INIT = "nndsvd"
NMF_MAX_ITER = 500
NMF_RANDOM_STATE = 42

# ---- EXPIMAP (scArches) — pathway-mask conditional VAE ----
# 10 monocyte-relevant HALLMARK terms; mask built by symbol->Ensembl translation
# from lib1_immune.gmt. Latent dim = number of surviving terms (target 10).
EXPIMAP_MASK_TERMS = [
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE",
    "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "HALLMARK_IL6_JAK_STAT3_SIGNALING",
    "HALLMARK_COMPLEMENT",
    "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
    "HALLMARK_GLYCOLYSIS",
    "HALLMARK_MTORC1_SIGNALING",
    "HALLMARK_HYPOXIA",
]
EXPIMAP_SOURCE_GMT = LIB1_GMT
EXPIMAP_MASK_GMT = NB_DIR / f"{ADATA_PATH.stem}_expimap_hallmark.gmt"
EXPIMAP_N_EPOCHS = 400
EXPIMAP_ALPHA = 0.7
EXPIMAP_ALPHA_KL = 0.5
EXPIMAP_ALPHA_EPOCH_ANNEAL = 130
EXPIMAP_KWARGS = dict(
    condition_key="donor_id",      # NB recon path requires batch one-hot (scarches/models/expimap/expimap.py:194)
    recon_loss="nb",               # requires raw counts in adata.X (true after HVG block)
    hidden_layer_sizes=(256, 256),
)
EXPIMAP_CACHE_DIR = MODEL_CACHE_DIR / "expimap_k10" / _semantic_cache_slug(
    {
        **EXPIMAP_KWARGS,
        "terms": tuple(EXPIMAP_MASK_TERMS),
        "alpha": EXPIMAP_ALPHA,
        "alpha_kl": EXPIMAP_ALPHA_KL,
        "alpha_epoch_anneal": EXPIMAP_ALPHA_EPOCH_ANNEAL,
    },
    EXPIMAP_N_EPOCHS, 0, EXPIMAP_ALPHA_EPOCH_ANNEAL, HVG_TOP_N,
)
print(f"expimap_k10     cache_dir: {EXPIMAP_CACHE_DIR}")

# ---- Benchmark / report ----
MODEL_NAMES = ["semantic_geom", "semantic_genept", "ldvae_nn", "schpf_k10", "nmf_k10", "expimap_k10"]
PER_PROJECTION_N_TOP = 50
CLUSTER_N_TOP = 500     # pool of top-loaded genes fed into UMAP+Leiden and hierarchical clustering
GENE_MAPPING = ("feature_id", "feature_name")  # Ensembl → symbol via adata.var columns
ENABLE_LLM_GRADING = True  # set False to skip Sonnet/Haiku scoring

In [ ]:
import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import scanpy as sc
import torch
from sklearn.decomposition import NMF
from scipy.sparse import coo_matrix, issparse
import numpy as np
import schpf

# scArches 0.6.1 imports `from anndata import read`, which newer anndata dropped.
# Shim it before importing scarches — read_h5ad is the modern replacement.
import anndata as _ad
if not hasattr(_ad, "read"):
    _ad.read = _ad.read_h5ad

# Lazy install: scArches isn't a default project dep; only needed for the expimap_k10 baseline.
try:
    import scarches as sca
except ImportError:
    %pip install --quiet scarches==0.6.1
    import scarches as sca
print(f"scarches {sca.__version__}")

from benchmark_helpers import (
    NMFWrapper,
    _ExpimapAdapter,
    _ScviAdapter,
    build_expimap_mask_gmt,
    build_report,
    get_or_build_geneformer_map,
    get_or_build_genept_map,
    train_or_load_expimap,
    train_or_load_nonneg_ldvae,
    train_or_load_pickle,
    train_or_load_semantic_scvi,
)
from benchmarking import SemanticBenchmark
from train_schpf import train_nmf_model, train_schpf_model

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"NB_DIR = {NB_DIR}")


In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print("adata (raw):", adata.shape)
print("var cols:", list(adata.var.columns))

# Build/load the FULL-gene semantic maps for both priors first, then HVG-subset all of them together.
semantic_map_geneformer = get_or_build_geneformer_map(adata, SEMANTIC_CACHE_GENEFORMER)
print("semantic_map (geneformer, raw):", tuple(semantic_map_geneformer.shape))

semantic_map_genept = get_or_build_genept_map(adata, SEMANTIC_CACHE_GENEPT, GENEPT_PICKLE_PATH)
print("semantic_map (genept, raw):", tuple(semantic_map_genept.shape))

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(
        adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False
    )
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    keep_t = torch.as_tensor(keep)
    semantic_map_geneformer = semantic_map_geneformer[keep_t]
    semantic_map_genept     = semantic_map_genept[keep_t]
    print(f"After HVG filter (top {HVG_TOP_N}, flavor={HVG_FLAVOR}):")
    print("  adata:", adata.shape)
    print("  semantic_map (geneformer):", tuple(semantic_map_geneformer.shape))
    print("  semantic_map (genept):    ", tuple(semantic_map_genept.shape))
else:
    print("HVG filter skipped (HVG_TOP_N=None)")

In [ ]:
# Each scvi model writes a UUID into adata.uns, so each needs its own copy.
adata_sem_geneformer = adata.copy()
semantic_geneformer_model = train_or_load_semantic_scvi(
    adata_sem_geneformer,
    semantic_map_geneformer,
    cache_dir=SEMANTIC_GENEFORMER_CACHE_DIR,
    force_train=FORCE_TRAIN_SEMANTIC_GENEFORMER,
    max_epochs=SEMANTIC_GENEFORMER_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    **SEMANTIC_GENEFORMER_KWARGS,
)

adata_sem_genept = adata.copy()
semantic_genept_model = train_or_load_semantic_scvi(
    adata_sem_genept,
    semantic_map_genept,
    cache_dir=SEMANTIC_GENEPT_CACHE_DIR,
    force_train=FORCE_TRAIN_SEMANTIC_GENEPT,
    max_epochs=SEMANTIC_GENEPT_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_GENEPT_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_GENEPT_N_EPOCHS_KL_WARMUP,
    **SEMANTIC_GENEPT_KWARGS,
)

adata_ldvae = adata.copy()
ldvae_model = train_or_load_nonneg_ldvae(
    adata_ldvae,
    cache_dir=MODEL_CACHE_DIR / "ldvae_nn",
    force_train=FORCE_TRAIN_LDVAE,
    max_epochs=LDVAE_MAX_EPOCHS,
    **LDVAE_KWARGS,
)

schpf_model = train_or_load_pickle(
    "scHPF",
    lambda: train_schpf_model(adata, n_factors=N_FACTORS),
    cache_path=MODEL_CACHE_DIR / "schpf_k10.pkl",
    force_train=FORCE_TRAIN_SCHPF,
)

def _train_nmf():
    print(f"Training cNMF (sklearn NMF, k={N_FACTORS}, init={NMF_INIT})...")
    X_nmf = adata.X
    if issparse(X_nmf):
        if (X_nmf.data < 0).any():
            X_nmf.data = np.maximum(X_nmf.data, 0)
    else:
        X_nmf = np.maximum(X_nmf, 0)
    _nmf = NMF(
        n_components=N_FACTORS,
        init=NMF_INIT,
        random_state=NMF_RANDOM_STATE,
        max_iter=NMF_MAX_ITER,
    )
    W = _nmf.fit_transform(X_nmf)
    H = _nmf.components_
    return NMFWrapper(model=_nmf, W=W, H=H, feature_names=adata.var_names)

nmf_model = train_or_load_pickle(
    "cNMF/NMF",
    _train_nmf,
    cache_path=MODEL_CACHE_DIR / "nmf_k10.pkl",
    force_train=FORCE_TRAIN_NMF,
)

# ---- EXPIMAP (pathway-mask conditional VAE) ----
# Build the symbol->Ensembl-translated mask GMT once per adata (cached on disk).
if not EXPIMAP_MASK_GMT.exists():
    build_expimap_mask_gmt(
        adata, EXPIMAP_SOURCE_GMT, EXPIMAP_MASK_TERMS, EXPIMAP_MASK_GMT,
    )
adata_expimap = adata.copy()
expimap_model = train_or_load_expimap(
    adata_expimap,
    gmt_path=EXPIMAP_MASK_GMT,
    cache_dir=EXPIMAP_CACHE_DIR,
    force_train=FORCE_TRAIN_EXPIMAP,
    n_epochs=EXPIMAP_N_EPOCHS,
    alpha_kl=EXPIMAP_ALPHA_KL,
    alpha=EXPIMAP_ALPHA,
    alpha_epoch_anneal=EXPIMAP_ALPHA_EPOCH_ANNEAL,
    **EXPIMAP_KWARGS,
)
print(f"expimap_k10 active terms: {len(expimap_model.nonzero_terms())} / {len(adata_expimap.uns['terms'])}")


In [ ]:
# ============================================================
# Training-loss curves — train/val for each SemanticSCVI variant and LDVAE+.
# scHPF / cNMF have no per-epoch history, so they're skipped.
# EXPIMAP: attempted with semantic=False; plot_training_curves silently skips
# if scArches' history dict doesn't match scvi's {metric: DataFrame} schema.
# Filenames are matched by build_report (training_curves_semantic.png and
# training_curves_semantic_genept.png get separate panels in the final HTML).
# ============================================================
import importlib, benchmark_helpers
importlib.reload(benchmark_helpers)
from benchmark_helpers import plot_training_curves

plot_training_curves(
    semantic_geneformer_model, "SemanticSCVI (Geneformer)",
    OUT_DIR / "training_curves_semantic.png",
    semantic=True,
)
plot_training_curves(
    semantic_genept_model, "SemanticSCVI (GenePT)",
    OUT_DIR / "training_curves_semantic_genept.png",
    semantic=True,
)
plot_training_curves(
    ldvae_model, "LDVAE+",
    OUT_DIR / "training_curves_ldvae.png",
    semantic=False,
)
plot_training_curves(
    expimap_model, "EXPIMAP",
    OUT_DIR / "training_curves_expimap.png",
    semantic=False,
)
print("scHPF / cNMF: no per-epoch training history (skipping).")


In [ ]:
models = {
    MODEL_NAMES[0]: _ScviAdapter(semantic_geneformer_model, adata_sem_geneformer),
    MODEL_NAMES[1]: _ScviAdapter(semantic_genept_model,     adata_sem_genept),
    MODEL_NAMES[2]: _ScviAdapter(ldvae_model, adata_ldvae),
    MODEL_NAMES[3]: schpf_model,
    MODEL_NAMES[4]: nmf_model,
    MODEL_NAMES[5]: _ExpimapAdapter(expimap_model, adata_expimap),
}
for name, m in models.items():
    z = m.get_latent_representation()
    print(f"{name}: latent shape = {z.shape}")


In [ ]:
bench = SemanticBenchmark(
    models,
    adata,
    pathway_index=str(PATHWAY_INDEX),
    gene_mapping=GENE_MAPPING,
    out_dir=str(OUT_DIR),
)
# semantic_alignment is computed against a single reference embedding. We pass the
# Geneformer prior here (matches the original notebook) so the score is identical
# for ldvae/schpf/nmf/semantic_geom; semantic_genept will be scored against a
# different prior than its own — interpret accordingly.
bench.run_figures(
    semantic_map=semantic_map_geneformer,
    lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
    lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
    per_projection_n_top=PER_PROJECTION_N_TOP,
    cluster_n_top=CLUSTER_N_TOP,
)

In [ ]:
if ENABLE_LLM_GRADING:
    bench.run_grading(
        lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
        lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
        per_projection_n_top=PER_PROJECTION_N_TOP,
        cluster_n_top=CLUSTER_N_TOP,
    )
else:
    print("LLM grading skipped (ENABLE_LLM_GRADING=False)")

In [ ]:
# ============================================================
# Cache latent projections (+ denoised_gamma from semantic_genept) to AnnData
# for downstream (e.g. pyimpulse) analysis.
# Each model's latent goes into obsm["X_<model_name>"].
# denoised_gamma layer + model_dispersions in uns come from semantic_genept.
# ============================================================
import types

adata_proj = adata.copy()
for name, m in models.items():
    z = np.asarray(m.get_latent_representation())
    adata_proj.obsm[f"X_{name}"] = z
    print(f"{name}: obsm['X_{name}'] = {z.shape}")

# Patch get_normalized_expression: LDVAE.generative() uses **kwargs which hides
# transform_batch from inspect.signature() -> base RNASeqMixin raises
# NotImplementedError. Mirror pyimpulse's _patch_get_normalized_expression.
def _fixed_get_transform_batch_gen_kwargs(self, batch):
    return {"transform_batch": batch}

semantic_genept_model._get_transform_batch_gen_kwargs = types.MethodType(
    _fixed_get_transform_batch_gen_kwargs, semantic_genept_model
)

# Denoised expression + dispersions from SemanticSCVI (GenePT prior).
denoised = np.asarray(
    semantic_genept_model.get_normalized_expression(
        adata_sem_genept, library_size=10_000
    )
)
adata_proj.layers["denoised_gamma"] = denoised
print(f"layers['denoised_gamma'] = {denoised.shape} (from semantic_genept)")

try:
    px_r = semantic_genept_model.module.px_r.detach().exp().cpu().numpy()
    adata_proj.uns["model_dispersions"] = px_r
    print(f"uns['model_dispersions'] = {px_r.shape}")
except AttributeError:
    print("WARN: could not extract px_r dispersions from semantic_genept")

projections_path = NB_DIR / f"{ADATA_PATH.stem}_projections.h5ad"
adata_proj.write_h5ad(projections_path)
print(f"Projections written: {projections_path}")


In [ ]:
from datetime import datetime

notes = (
    f"SemanticSCVI [Geneformer]: {SEMANTIC_GENEFORMER_KWARGS['loss_mode']} loss, "
    f"cw={SEMANTIC_GENEFORMER_KWARGS['coherence_weight']}, "
    f"warmup={SEMANTIC_GENEFORMER_WARMUP_EPOCHS}/{SEMANTIC_GENEFORMER_MAX_EPOCHS} ep. "
    f"SemanticSCVI [GenePT-3-large]: {SEMANTIC_GENEPT_KWARGS['loss_mode']} loss, "
    f"cw={SEMANTIC_GENEPT_KWARGS['coherence_weight']}, "
    f"warmup={SEMANTIC_GENEPT_WARMUP_EPOCHS}/{SEMANTIC_GENEPT_MAX_EPOCHS} ep. "
    f"LDVAE+: weights_positive=True, {LDVAE_MAX_EPOCHS} ep. "
    f"scHPF: k={N_FACTORS}. cNMF: sklearn NMF k={N_FACTORS}, init={NMF_INIT}. "
    f"EXPIMAP: {len(EXPIMAP_MASK_TERMS)} HALLMARK terms, alpha={EXPIMAP_ALPHA}, "
    f"alpha_kl={EXPIMAP_ALPHA_KL}, {EXPIMAP_N_EPOCHS} ep."
)
report_path = build_report(OUT_DIR, MODEL_NAMES, adata.shape, notes=notes)

# Rename to <dataset>_<YYYYMMDD_HHMMSS>.html and drop intermediate artifacts.
# Preserved: LLM/score caches (`_llm_cache/`, `_score_cache/`) and prior .html reports.
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
final_name = f"{ADATA_PATH.stem}_{ts}.html"
final_path = OUT_DIR / final_name
report_path.rename(final_path)

PRESERVE_DIRS = {"_llm_cache", "_score_cache"}
for child in OUT_DIR.iterdir():
    if child == final_path:
        continue
    if child.is_dir():
        if child.name in PRESERVE_DIRS:
            continue
        import shutil
        shutil.rmtree(child)
    else:
        if child.suffix == ".html":
            continue
        child.unlink()

print(f"Report written: {final_path}")
print(f"Size: {final_path.stat().st_size / 1e6:.1f} MB")
